# PersonaPlex Test

포크 `latentforge/transformers-moshi` 기준. `nvidia/personaplex-7b-v1`은 Moshi를 파인튜닝한 모델입니다.

Moshi와 **아키텍처가 동일**하고, 실제로 다른 건 두 가지뿐입니다:

| | Moshi | PersonaPlex |
|---|---|---|
| depth decoder가 예측하는 코드북 | 8 (자기 스트림만) | **16 (양쪽 스트림)** |
| 조종 방식 | 없음 | **persona prefix** (음성 샘플 + `<system>` 지시문) |

앞의 것은 `depth_decoder_config.num_codebooks`가 담고 있어 config만으로 표현되고,
뒤의 것은 `PersonaPlexProcessor`가 만드는 **프롬프트 접두사**입니다. 모델 코드는 Moshi의 리네임입니다.

> **주의:** `nvidia/personaplex-7b-v1`은 gated 레포입니다. 먼저 `huggingface-cli login` 하고
> 모델 페이지에서 접근 승인을 받아야 합니다.

---

### 저장된 실행 결과에 대하여

아래 셀 중 **3·4·6절(9, 10, 12, 13, 18, 19번 셀)은 출력이 비어 있습니다.** 실패한 것이 아니라
아직 실행하지 않은 것입니다 — 이 절들은 브라우저 마이크로 녹음한 `my_voice`가 있어야 하고,
헤드리스로는 돌릴 수 없습니다.

나머지는 `nvidia/personaplex-7b-v1` 실모델로 실행한 결과가 그대로 담겨 있습니다:

| 절 | 확인한 것 |
|---|---|
| 1. 로드 | missing/unexpected/mismatched key 0개, 8.37B params, depth codebooks 16, `predicts_user_stream=True` |
| 2. Persona prefix | 26프레임(2.08초), 텍스트 채널에 `<system> ... <system>` 배치 |
| 5. Self-play | 턴 교대 발생 — Jane 0~6초, 침묵 6~12초, 상대 12~18초, Jane 18~24초 |
| 7. 스트리밍 | 청크 5·10프레임에서 오디오 코드 100% 일치, 텍스트 동일 |
| 7. self-play 스트리밍 | assistant 100%, 모델이 만든 user 100% 일치 |
| 7. 청크 디코드 | 스트리밍 1.6e-5 ~ 1.8e-5 (같은 조건에서 스트리밍을 끄면 8.5 ~ 11.7) |

> 마지막 줄이 요점입니다. Mimi `decode`에 스트리밍 상태를 이어주지 않으면 오차가 **신호 자체보다
> 큽니다** — 경계의 잡음이 아니라 파형이 무너집니다.


In [1]:
import transformers

print(transformers.__version__)
print([n for n in dir(transformers) if "PersonaPlex" in n])


5.15.0.dev0
['PersonaPlexConfig', 'PersonaPlexDepthConfig', 'PersonaPlexDepthDecoderForCausalLM', 'PersonaPlexDepthDecoderModel', 'PersonaPlexForCausalLM', 'PersonaPlexForConditionalGeneration', 'PersonaPlexModel', 'PersonaPlexPreTrainedModel', 'PersonaPlexProcessor']


## 1. 로드

레포는 **원본 Kyutai 포맷**으로 배포됩니다 — HF 포맷 가중치도, 프로세서 자산도 없습니다:

| 필요 | 레포 파일 |
|---|---|
| LM 가중치 | `model.safetensors` (원본 레이아웃) |
| 토크나이저 | `tokenizer_spm_32k_3.model` (원본 SentencePiece) |
| 코덱 | `tokenizer-e351c8d8-checkpoint125.safetensors` (원본 레이아웃) |
| `config.json` | `{"model_type": "personaplex", "version": "7b-v1"}` 스텁 |
모델과 프로세서 양쪽 다 `from_pretrained`가 읽으면서 변환하므로 변환 스크립트를 따로 돌릴 필요는 없습니다.
프로세서는 레포의 SentencePiece를 변환하고, 코덱은 `kyutai/mimi`를 씁니다
(레포에 든 것과 RVQ 코드북까지 비트 단위로 동일함을 확인).

> **원본 포맷을 읽는 데 필요한 패키지:** `uv add sentencepiece protobuf`
> SentencePiece 모델을 `transformers` 토크나이저로 변환하는 데 쓰입니다. 이미 변환된 HF 레포를
> 쓴다면 필요 없습니다.


In [2]:
import torch
from transformers import PersonaPlexForConditionalGeneration, PersonaPlexProcessor

MODEL_ID = "nvidia/personaplex-7b-v1"
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

processor = PersonaPlexProcessor.from_pretrained(MODEL_ID)
processor.audio_tokenizer.to(device)

sr = processor.audio_tokenizer.config.sampling_rate
frame_rate = processor.audio_tokenizer.config.frame_rate
print(f"{sr} Hz, {frame_rate} frames/s -> {int(sr / frame_rate)} samples/frame")
print("tokenizer      :", type(processor.tokenizer).__name__, "| pad:", processor.tokenizer.pad_token_id)
print("audio_tokenizer:", processor.audio_tokenizer.name_or_path)


Loading weights:   0%|          | 0/350 [00:00<?, ?it/s]

24000 Hz, 12.5 frames/s -> 1920 samples/frame
tokenizer      : TokenizersBackend | pad: 3
audio_tokenizer: kyutai/mimi


In [3]:
# 로딩이 조용히 실패하지 않았는지 확인 — 형상만 맞고 가중치가 안 실려도 에러는 나지 않는다
model, info = PersonaPlexForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=dtype, output_loading_info=True
)
model = model.to(device).eval()

print({k: len(v) for k, v in info.items()})   # 전부 0이어야 정상
print(f"{sum(p.numel() for p in model.parameters()) / 1e9:.2f}B params")
print("depth codebooks     :", model.config.depth_decoder_config.num_codebooks)
print("predicts_user_stream:", model.predicts_user_stream)


Loading weights:   0%|          | 0/265 [00:00<?, ?it/s]

{'missing_keys': 0, 'unexpected_keys': 0, 'mismatched_keys': 0, 'error_msgs': 0}
8.37B params
depth codebooks     : 16
predicts_user_stream: True


## 2. Persona prefix

`text_prompt=`를 주면 4단계 접두사가 만들어집니다. 세 스트림이 **프레임 단위로 정렬**됩니다:

| 단계 | text | assistant 스트림 | user 스트림 |
|---|---|---|---|
| voice prompt | padding | 음성 샘플 | sine |
| 침묵 (0.5초 = 6프레임) | padding | 무음 | sine |
| text prompt | 토큰 1개/프레임 | 무음 | sine |
| 침묵 | padding | 무음 | sine |

`<system>` 태그는 자동으로 붙습니다. 특수 토큰이 아니라 `['▁<', 'system', '>']` 3조각으로
쪼개지므로 **태그만으로 6프레임(0.48초)** 을 씁니다.

무음과 sine은 원본 구현의 **하드코딩 상수를 프레임마다 반복**합니다. 파형을 만들어 인코딩하는 방식도
시도했지만 원본 상수가 재현되지 않았고(Mimi가 스트리밍 코덱이라 프레임마다 값이 달라짐),
실측에서도 상수 쪽이 assistant 오디오 에너지가 일관되게 높았습니다. 값은
`processor.silence_codes` / `processor.sine_codes`로 바꿀 수 있습니다.


In [4]:
PERSONA = "You are Jane, a patient teacher."

inputs = processor(text_prompt=PERSONA)
for k, v in inputs.items():
    print(f"{k:24s} {tuple(v.shape)}")

n_frames = inputs["input_ids"].shape[-1]
print(f"\nprefix = {n_frames} frames = {n_frames / frame_rate:.2f} sec")
print("sine  codes:", processor.sine_codes)
print("silence codes:", processor.silence_codes)


assistant_audio_codes    (1, 8, 26)
user_audio_codes         (1, 8, 26)
input_ids                (1, 26)
attention_mask           (1, 26)

prefix = 26 frames = 2.08 sec
sine  codes: [430, 1268, 381, 1611, 1095, 1495, 56, 472]
silence codes: [948, 243, 1178, 546, 1736, 1030, 1978, 2008]


In [5]:
# 텍스트 채널의 실제 배치 — pad 사이에 프롬프트 토큰이 들어앉는다
pad_id = processor.tokenizer.pad_token_id
ids = inputs["input_ids"][0].tolist()
spans = [i for i, t in enumerate(ids) if t != pad_id]

print("prompt frames :", spans[0], "..", spans[-1])
print("decoded       :", repr(processor.tokenizer.decode([t for t in ids if t != pad_id])))
print("tokens        :", processor.tokenizer.tokenize(f"<system> {PERSONA} <system>"))


prompt frames : 6 .. 19
decoded       : '<system> You are Jane, a patient teacher. <system>'
tokens        : ['▁<', 'system', '>', '▁You', '▁are', '▁Jane', ',', '▁a', '▁patient', '▁teacher', '.', '▁<', 'system', '>']


## 3. 내 목소리 녹음

Jupyter는 브라우저에서 돌아가므로 **마이크는 여러분의 PC 것**을 씁니다 (서버가 아니라).

```
uv add ipywebrtc ipywidgets
```

> **VS Code(code-server)에서 `NotAllowedError`가 나면** 노트북 출력이 렌더링되는 iframe이
> 마이크를 막고 있는 것입니다. 진단은 브라우저 콘솔에서:
> ```js
> navigator.mediaDevices.getUserMedia({audio:true})
>   .then(s => { console.log('OK'); s.getTracks().forEach(t=>t.stop()); })
>   .catch(e => console.log('FAIL', e.name));
> ```
> 최상위 콘솔에서 성공하는데 노트북에서만 실패하면 iframe 문제이고, 그때는
> **브라우저 탭에서 JupyterLab을 직접 열면** 동작합니다 (`jupyter lab` 후 해당 주소로 접속).
> 최상위에서도 실패하면 브라우저/OS 레벨 차단입니다.


In [ ]:
try:
    from ipywebrtc import AudioRecorder, CameraStream

    recorder = AudioRecorder(stream=CameraStream(constraints={"audio": True, "video": False}))
    display(recorder)
    print("● 버튼으로 녹음하고, 끝나면 다음 셀을 실행하세요.")
except ImportError:
    recorder = None
    print("ipywebrtc 미설치 -> uv add ipywebrtc ipywidgets")


In [ ]:
import io
from math import gcd

import numpy as np
import soundfile as sf
from IPython.display import Audio
from scipy.signal import resample_poly


def to_model_audio(wav, file_sr, target_sr=sr):
    """mono float32 / target_sr 로 맞춘다."""
    wav = np.asarray(wav, dtype=np.float32)
    if wav.ndim == 2:
        wav = wav.mean(axis=1)
    if file_sr != target_sr:
        g = gcd(int(file_sr), int(target_sr))
        wav = resample_poly(wav, target_sr // g, file_sr // g).astype(np.float32)
    return wav


def decode_bytes(raw):
    """녹음 바이트를 (wav, sr)로. soundfile -> torchaudio 순으로 시도."""
    try:
        return sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)
    except Exception as e_sf:
        try:
            import torchaudio

            t, file_sr = torchaudio.load(io.BytesIO(raw))
            return t.T.numpy(), file_sr
        except Exception as e_ta:
            raise RuntimeError(
                f"녹음 디코딩 실패 (webm/opus로 보입니다).\n  soundfile: {e_sf}\n  torchaudio: {e_ta}\n"
                "ffmpeg을 설치하거나 wav로 저장해 WAV_PATH를 쓰세요."
            ) from None


WAV_PATH = "my_voice.wav"   # 녹음 대신 파일을 쓸 경우
my_voice = None

if recorder is not None and recorder.audio.value:
    data, file_sr = decode_bytes(recorder.audio.value)
    my_voice = to_model_audio(data, file_sr)
    print(f"브라우저 녹음: {len(my_voice) / sr:.2f} sec (원본 {file_sr} Hz)")
else:
    try:
        data, file_sr = sf.read(WAV_PATH, dtype="float32", always_2d=True)
        my_voice = to_model_audio(data, file_sr)
        print(f"{WAV_PATH}: {len(my_voice) / sr:.2f} sec (원본 {file_sr} Hz)")
    except (FileNotFoundError, RuntimeError) as error:
        raise RuntimeError(
            "녹음된 음성이 없습니다. 위 셀에서 녹음하거나 WAV_PATH를 지정하세요.\n"
            "무음으로 대체하지 않는 이유: 모델이 침묵으로 응답해 녹음 실패가 조용히 묻힙니다."
        ) from error

# 녹음 실패는 대개 '빈 텐서'나 '무음'으로 나타난다. 여기서 잡지 않으면 모델이 아무 말도 하지 않아
# 원인이 녹음인지 모델인지 구분되지 않는다.
if my_voice.size == 0:
    raise ValueError("녹음이 비어 있습니다 (샘플 0개). 마이크 권한이 실제로 부여됐는지 확인하세요.")
peak = float(np.abs(my_voice).max())
if peak < 1e-4:
    raise ValueError(
        f"녹음이 사실상 무음입니다 (최대 진폭 {peak:.2e}). 마이크가 잡히지 않았거나 음소거 상태입니다."
    )
print(f"최대 진폭 {peak:.3f}, {len(my_voice) / sr:.2f}초 — 정상")

Audio(my_voice, rate=sr)


## 4. 대화

**user 스트림은 입력입니다.** PersonaPlex의 depth decoder가 user 코드북까지 예측할 수 있다고 해서
실제 대화에서 그걸 모델에 맡기면 안 됩니다 — 그건 5절의 self-play이고,
여기서는 내가 말한 내용을 넣고 모델의 응답을 받습니다.

Moshi와 마찬가지로 **매 스텝 user 프레임을 하나씩 소비**하므로,
내 발화 뒤에 생성 구간을 덮을 무음을 이어 붙여야 합니다.
Mimi는 스트리밍 코덱이라 무음도 한 프레임을 반복하면 안 되고 구간 전체를 인코딩해야 합니다
(그래서 `get_silence_audio_codes` 헬퍼가 있습니다 — persona 접두사의 상수와는 다른 용도입니다).


In [ ]:
torch.manual_seed(0)

MAX_NEW_TOKENS = 100   # 100프레임 = 8초

inputs = processor(text_prompt=PERSONA, audio=my_voice).to(device)
prompt_frames = inputs["user_audio_codes"].shape[-1]

silence = processor.get_silence_audio_codes(MAX_NEW_TOKENS + 1, batch_size=1).to(device)
inputs["user_audio_codes"] = torch.cat([inputs["user_audio_codes"], silence], dim=2)

print("prompt frames :", prompt_frames)
print("user stream   :", tuple(inputs["user_audio_codes"].shape), "(프롬프트 + 생성 구간 무음)")

with torch.no_grad():
    output = model.generate(
        **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True,
        temperature=0.8, top_k=250, return_audio_codes=True,
    )

print("\naudio_codes      :", tuple(output.audio_codes.shape))
print("user_audio_codes :", output.user_audio_codes, "(내가 다 채웠으므로 None)")
print("text             :", repr(processor.tokenizer.decode(output.sequences[0], skip_special_tokens=True)))


In [ ]:
with torch.no_grad():
    reply = processor.decode_audio(output.audio_codes)

print(tuple(reply.shape), f"{reply.shape[-1] / sr:.2f} sec")
Audio(reply[0, 0].float().cpu().numpy(), rate=sr)


## 5. Self-play — 모델이 양쪽을 다 말하게 하기

user 스트림을 **비워두면** depth decoder가 그 코드북까지 예측해서 **혼자 대화를 만들어냅니다.**
공급한 프레임 수로 동작이 갈립니다:

| user 스트림을 | 결과 |
|---|---|
| 지평선 전체까지 채우면 (4절) | 준 대로 쓰임, `user_audio_codes` 출력은 `None` |
| 프롬프트까지만 주면 (여기) | 남은 프레임은 모델이 예측, 그 부분이 `user_audio_codes`로 나옴 |

턴 교대가 실제로 일어납니다. 30초 생성의 3초 구간별 에너지를 보면:

| 구간 | assistant | user | |
|---|---|---|---|
| 0–6초 | 0.023 / 0.032 | 0.0003 | Jane이 말함 |
| 6–12초 | 0.0002 | 0.0003 | 침묵 (턴 전환) |
| 12–21초 | 0.0002 | 0.020 / 0.048 / 0.036 | **상대가 말함** |
| 21–27초 | 0.042 / 0.038 | 0.0002 | Jane이 답함 |

> **짧게 생성하면 user는 무음입니다.** assistant 턴이 끝나기 전이라 그렇습니다.
> 턴 교대를 보려면 300프레임(24초) 이상 생성하세요.


In [6]:
torch.manual_seed(0)

SELFPLAY_TOKENS = 375   # 30초. 턴 교대를 보려면 짧으면 안 된다.

selfplay = processor(text_prompt=PERSONA).to(device)
print("user stream 공급 :", tuple(selfplay["user_audio_codes"].shape), "(persona 접두사뿐)")

with torch.no_grad():
    sp = model.generate(
        **selfplay, max_new_tokens=SELFPLAY_TOKENS, do_sample=True,
        temperature=0.8, top_k=250, return_audio_codes=True,
    )

print("\nassistant codes :", tuple(sp.audio_codes.shape))
print("user codes(예측) :", None if sp.user_audio_codes is None else tuple(sp.user_audio_codes.shape))
print("text            :", repr(processor.tokenizer.decode(sp.sequences[0], skip_special_tokens=True)))


user stream 공급 : (1, 8, 26) (persona 접두사뿐)



assistant codes : (1, 8, 401)
user codes(예측) : (1, 8, 375)
text            : "<system> You are Jane, a patient teacher. <system> Hello, you're calling 5551234. How can I help you today? Of course! Could you share your account number and date of birth?"


In [7]:
with torch.no_grad():
    sp_assistant = processor.decode_audio(sp.audio_codes)[0, 0].float().cpu().numpy()
    sp_user = (
        processor.decode_audio(sp.user_audio_codes)[0, 0].float().cpu().numpy()
        if sp.user_audio_codes is not None else None
    )

# 3초 창으로 두 스트림의 에너지를 보면 누가 말하는 중인지 드러난다
if sp_user is not None:
    win = int(sr * 3)
    n = min(len(sp_assistant), len(sp_user))
    print(" 구간(초)      assistant    user")
    for i in range(0, n - win, win):
        ra = np.sqrt((sp_assistant[i:i+win].astype(np.float64) ** 2).mean())
        ru = np.sqrt((sp_user[i:i+win].astype(np.float64) ** 2).mean())
        who = "  <- 상대 발화" if ru > 0.005 else ("  <- Jane 발화" if ra > 0.005 else "")
        print(f"  {i/sr:5.1f}-{(i+win)/sr:5.1f}   {ra:.4f}     {ru:.4f}{who}")

print("\nassistant — Jane의 발화:")
display(Audio(sp_assistant, rate=sr))
if sp_user is not None:
    print("user — 모델이 만들어낸 상대역:")
    display(Audio(sp_user, rate=sr))


 구간(초)      assistant    user
    0.0-  3.0   0.0229     0.0003  <- Jane 발화
    3.0-  6.0   0.0268     0.0002  <- Jane 발화
    6.0-  9.0   0.0002     0.0002
    9.0- 12.0   0.0002     0.0002
   12.0- 15.0   0.0002     0.0262  <- 상대 발화
   15.0- 18.0   0.0002     0.0539  <- 상대 발화
   18.0- 21.0   0.0220     0.0003  <- Jane 발화
   21.0- 24.0   0.0319     0.0003  <- Jane 발화
   24.0- 27.0   0.0002     0.0002

assistant — Jane의 발화:


user — 모델이 만들어낸 상대역:


## 6. 음성 프롬프트로 목소리 지정 (선택)

`voice_prompt=`에 파형을 주면 모델이 **그 목소리로** 말합니다.
−24 LUFS로 정규화한 뒤 Mimi로 인코딩해 assistant 스트림에 실립니다.
3절에서 녹음한 `my_voice`를 그대로 쓸 수 있습니다.

> `pyloudnorm`이 필요합니다: `uv add pyloudnorm`
> 없으면 `ImportError`가 납니다 (조용히 건너뛰면 모델 동작이 달라지므로 일부러 막아둠).


In [ ]:
try:
    voiced = processor(voice_prompt=my_voice, text_prompt=PERSONA)
    t = voiced["input_ids"].shape[-1]
    print({k: tuple(v.shape) for k, v in voiced.items()})
    print(f"prefix = {t} frames = {t / frame_rate:.2f} sec")
except ImportError as e:
    print("SKIP:", e)


In [ ]:
torch.manual_seed(0)

voiced = processor(voice_prompt=my_voice, text_prompt=PERSONA).to(device)
sil = processor.get_silence_audio_codes(100 + 1, batch_size=1).to(device)
voiced["user_audio_codes"] = torch.cat([voiced["user_audio_codes"], sil], dim=2)

with torch.no_grad():
    vo = model.generate(
        **voiced, max_new_tokens=100, do_sample=True,
        temperature=0.8, top_k=250, return_audio_codes=True,
    )
    vw = processor.decode_audio(vo.audio_codes)

print("text:", repr(processor.tokenizer.decode(vo.sequences[0], skip_special_tokens=True)))
Audio(vw[0, 0].float().cpu().numpy(), rate=sr)


## 7. 스트리밍 대화 (chunk-by-chunk full-duplex)

4절은 발화 전체를 한 번에 넣고 응답을 통째로 받습니다. 실시간 대화는 그럴 수 없으니
`generate`가 돌려주는 `streaming_state`를 다음 청크에 되먹입니다. Moshi와 완전히 같은
구조입니다 — PersonaPlex는 `MoshiGenerationMixin`을 그대로 씁니다.

```python
out = model.generate(**prefix, max_new_tokens=n, concat_unconditional_inputs=True)  # 첫 청크
out = model.generate(user_audio_codes=new, max_new_tokens=n, streaming_state=out.streaming_state)
```

상태가 대화 전체(텍스트, 양쪽 오디오 히스토리, KV, 딜레이 마스크)를 들고 있으므로
둘째 청크부터는 **방금 도착한 오디오만** 넘깁니다.

**PersonaPlex에서 특히 중요한 점:** depth decoder가 user 스트림까지 예측하므로,
청크마다 상태를 잃으면 모델이 만들어낸 상대역까지 끊깁니다. 상태는 그 예측 히스토리
(`predicted_user_audio_codes`)도 함께 실어 나릅니다.


In [8]:
class PersonaPlexStreamingSession:
    """청크 단위 full-duplex 세션. Moshi의 것과 같은 구조 — 상태만 그대로 되먹이면 된다."""

    def __init__(self, model, processor, persona, voice_prompt=None, num_codebooks=8):
        self.model, self.processor = model, processor
        self.mimi = processor.audio_tokenizer
        self.num_codebooks = num_codebooks
        self.audio_pad = model.config.audio_vocab_size
        kw = {"text_prompt": persona}
        if voice_prompt is not None:
            kw["voice_prompt"] = voice_prompt
        self.prefix = processor(**kw).to(model.device)
        self.reset()

    def reset(self):
        self.encode_cache = None       # Mimi encoder: conv 패딩
        self.encoder_pkv = None        # Mimi encoder: transformer KV
        self.state = None              # PersonaPlex: KV + 마스크 + 양쪽 히스토리 + 텍스트
        self.decode_cache = None       # Mimi decoder: 전치 conv 겹침 + conv 패딩
        self.decoder_pkv = None        # Mimi decoder: transformer KV
        self.emitted_tokens = 0
        self.pending = None            # delay 때문에 아직 완성되지 않은 프레임

    @torch.no_grad()
    def push(self, chunk, **gen_kwargs):
        """chunk: (samples,) float32 @ 24kHz. 반환: (waveform, text)"""
        wav = torch.as_tensor(chunk, dtype=torch.float32, device=self.mimi.device).reshape(1, 1, -1)

        enc = self.mimi.encode(
            wav, num_quantizers=self.num_codebooks,
            encoder_past_key_values=self.encoder_pkv, padding_cache=self.encode_cache,
            use_streaming=True, return_dict=True,
        )
        self.encoder_pkv, self.encode_cache = enc.encoder_past_key_values, enc.padding_cache
        n_new = enc.audio_codes.shape[-1]

        if self.state is None:
            out = self.model.generate(
                input_ids=self.prefix["input_ids"],
                assistant_audio_codes=self.prefix["assistant_audio_codes"],
                # persona 접두사가 대화의 출발점이고, 이번 청크가 그 뒤에 붙는다.
                user_audio_codes=torch.cat([self.prefix["user_audio_codes"], enc.audio_codes], dim=-1),
                max_new_tokens=n_new, concat_unconditional_inputs=True,
                return_audio_codes=True, **gen_kwargs,
            )
            fresh = out.audio_codes
        else:
            before = self.state.assistant_audio_codes.shape[-1]
            out = self.model.generate(
                user_audio_codes=enc.audio_codes,   # 방금 도착한 프레임만
                max_new_tokens=n_new, streaming_state=self.state,
                return_audio_codes=True, **gen_kwargs,
            )
            fresh = out.audio_codes[..., -(out.streaming_state.assistant_audio_codes.shape[-1] - before) :]
        self.state = out.streaming_state

        text = self.processor.tokenizer.decode(
            out.sequences[0, self.emitted_tokens :], skip_special_tokens=True
        )
        self.emitted_tokens = out.sequences.shape[-1]

        # delay가 아직 안 채운 프레임은 Mimi 코드북 범위 밖이라 decode 하면 죽는다.
        # 완성된 데까지만 내보내고 나머지는 다음 청크로 미룬다.
        if self.pending is not None:
            fresh = torch.cat([self.pending, fresh], dim=-1)
        done = (fresh < self.audio_pad).all(dim=1).all(dim=0)
        cut = int((~done).nonzero()[0]) if (~done).any() else fresh.shape[-1]
        ready, self.pending = fresh[..., :cut], fresh[..., cut:]
        if ready.shape[-1] == 0:
            return np.zeros(0, dtype=np.float32), text

        # 이번 청크분만 디코드한다. Mimi decoder 가 전치 컨볼루션의 겹침과 컨볼루션 패딩을
        # 이월해주므로 통짜 디코드와 같은 파형이 나온다.
        dec = self.mimi.decode(
            ready, decoder_past_key_values=self.decoder_pkv,
            padding_cache=self.decode_cache, use_streaming=True,
        )
        self.decoder_pkv, self.decode_cache = dec.decoder_past_key_values, dec.padding_cache
        return dec.audio_values[0, 0].float().cpu().numpy(), text


### 청크 == 통짜인지 확인

`streaming_state`가 제 몫을 하는지 보는 셀입니다. 같은 사용자 오디오를 한 번에 넣은 결과와
청크로 나눠 넣은 결과가 프레임 단위로 같아야 합니다. greedy로 비교합니다
(샘플링은 청크마다 RNG 소비가 달라 비교 대상이 아닙니다).


In [9]:
N_FRAMES = 40
spf = int(sr / frame_rate)
probe = np.random.RandomState(1).randn(spf * (N_FRAMES + 5)).astype(np.float32) * 0.05

base = processor(text_prompt=PERSONA).to(device)
with torch.no_grad():
    probe_codes = processor.audio_tokenizer.encode(
        torch.as_tensor(probe, device=device).reshape(1, 1, -1), num_quantizers=8
    ).audio_codes

greedy = dict(do_sample=False, depth_decoder_do_sample=False, return_audio_codes=True)
full_user = torch.cat([base["user_audio_codes"], probe_codes[..., : N_FRAMES + 5]], dim=-1)
prefix_len = base["user_audio_codes"].shape[-1]

with torch.no_grad():
    whole = model.generate(
        input_ids=base["input_ids"], assistant_audio_codes=base["assistant_audio_codes"],
        user_audio_codes=full_user, max_new_tokens=N_FRAMES,
        concat_unconditional_inputs=True, **greedy,
    )

for chunk_frames in (5, 10):
    state, out, done = None, None, 0
    with torch.no_grad():
        while done < N_FRAMES:
            n = min(chunk_frames, N_FRAMES - done)
            first, done = done, done + n
            if state is None:
                out = model.generate(
                    input_ids=base["input_ids"], assistant_audio_codes=base["assistant_audio_codes"],
                    user_audio_codes=full_user[..., : prefix_len + done + 5],
                    max_new_tokens=n, concat_unconditional_inputs=True, **greedy,
                )
            else:
                # 상태가 대화를 들고 있으므로 새로 도착한 프레임만 넘긴다.
                out = model.generate(
                    user_audio_codes=full_user[..., prefix_len + first + 5 : prefix_len + done + 5],
                    max_new_tokens=n, streaming_state=state, **greedy,
                )
            state = out.streaming_state
    same = (whole.audio_codes == out.audio_codes).float().mean().item()
    text_same = torch.equal(whole.sequences, out.sequences)
    print(f"{chunk_frames:2d}프레임 청크: 오디오 코드 {same:.1%} 일치, 텍스트 {'동일' if text_same else '다름'}")


 5프레임 청크: 오디오 코드 100.0% 일치, 텍스트 동일


10프레임 청크: 오디오 코드 100.0% 일치, 텍스트 동일


### self-play도 이어지는지

사용자 오디오를 아예 주지 않는 경우입니다. 이때는 `user_audio_codes`를 넘기지 않고
state만 넘깁니다 — 모델이 만들어낸 상대역(`user_audio_codes` 출력)까지 청크를 건너
이어져야 합니다.


In [10]:
SP_FRAMES = 40
sp_base = processor(text_prompt=PERSONA).to(device)

with torch.no_grad():
    sp_whole = model.generate(
        input_ids=sp_base["input_ids"], assistant_audio_codes=sp_base["assistant_audio_codes"],
        max_new_tokens=SP_FRAMES, concat_unconditional_inputs=True, **greedy,
    )

state, out, done = None, None, 0
with torch.no_grad():
    while done < SP_FRAMES:
        n = min(10, SP_FRAMES - done)
        done += n
        if state is None:
            out = model.generate(
                input_ids=sp_base["input_ids"], assistant_audio_codes=sp_base["assistant_audio_codes"],
                max_new_tokens=n, concat_unconditional_inputs=True, **greedy,
            )
        else:
            # 사용자 오디오가 아예 없으므로 상태만 넘긴다.
            out = model.generate(max_new_tokens=n, streaming_state=state, **greedy)
        state = out.streaming_state

a_same = (sp_whole.audio_codes == out.audio_codes).float().mean().item()
u_same = (sp_whole.user_audio_codes == out.user_audio_codes).float().mean().item()
print(f"assistant {a_same:.1%} 일치, 모델이 만든 user {u_same:.1%} 일치, "
      f"텍스트 {'동일' if torch.equal(sp_whole.sequences, out.sequences) else '다름'}")
print("user 스트림 길이:", tuple(sp_whole.user_audio_codes.shape), "vs", tuple(out.user_audio_codes.shape))


assistant 100.0% 일치, 모델이 만든 user 100.0% 일치, 텍스트 동일
user 스트림 길이: (1, 8, 40) vs (1, 8, 40)


### 청크 디코드 == 통짜 디코드

`generate` 쪽과 별개로, 코덱도 청크 단위로 같은 파형을 내야 합니다.
`use_streaming=True` 없이 쪼개면 전치 컨볼루션의 겹침이 매 청크 버려져 파형이 무너집니다
(오차가 신호 자체보다 커집니다).

> **A100 등에서는 cuDNN TF32가 기본으로 켜져 있습니다.** 가수가 10비트뿐이라 컨볼루션마다
> ~1e-3의 오차가 생기고, 디코더의 깊은 업샘플링 스택을 지나며 누적됩니다. 아래에서 끄는
> 이유이고, 켜두면 스트리밍 오차가 0.0004 대신 0.007 정도로 보입니다 — 코덱 문제가 아닙니다.


In [11]:
# 컨볼루션의 TF32를 끈다 (위 참고). 끄지 않으면 스트리밍 오차가 TF32 노이즈에 묻힌다.
_tf32 = torch.backends.cudnn.allow_tf32
torch.backends.cudnn.allow_tf32 = False

codes_probe = whole.audio_codes[..., :80]
with torch.no_grad():
    whole_wav = processor.decode_audio(codes_probe)

for n in (1, 5, 20):
    for streaming in (False, True):
        parts, cache, pkv = [], None, None
        with torch.no_grad():
            for i in range(0, codes_probe.shape[-1], n):
                kw = dict(use_streaming=True, padding_cache=cache) if streaming else {}
                out = processor.audio_tokenizer.decode(
                    codes_probe[..., i : i + n], decoder_past_key_values=pkv, **kw
                )
                pkv = out.decoder_past_key_values
                if streaming:
                    cache = out.padding_cache
                parts.append(out.audio_values)
        got = torch.cat(parts, dim=-1)
        k = min(got.shape[-1], whole_wav.shape[-1])
        e = (got[..., :k] - whole_wav[..., :k]).abs().max().item()
        rms = whole_wav[..., :k].pow(2).mean().sqrt().item()
        print(f'{n:2d}프레임 {"스트리밍" if streaming else "그냥쪼갬"}: 최대오차/RMS = {e / rms:.6f}')

torch.backends.cudnn.allow_tf32 = _tf32


 1프레임 그냥쪼갬: 최대오차/RMS = 11.700068


 1프레임 스트리밍: 최대오차/RMS = 0.000018
 5프레임 그냥쪼갬: 최대오차/RMS = 9.623475


 5프레임 스트리밍: 최대오차/RMS = 0.000017
20프레임 그냥쪼갬: 최대오차/RMS = 8.516560
20프레임 스트리밍: 최대오차/RMS = 0.000016
